# Estrutura tabular, limpeza e qualidade

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/02_estrutura_limpeza_e_qualidade.ipynb)

## 1. Uma variável por coluna, uma observação por linha

A estrutura tabular depende da unidade de análise. Valores atômicos e nomes de
campos estáveis facilitam seleção e junção, mas não determinam sozinhos a
ontologia correta. Tabelas separadas podem representar documentos, pessoas,
lugares e relações sem repetir tudo em uma única linha.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_03'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('openpyxl', 'openpyxl>=3.1,<4'), ('pypdf', 'pypdf>=5,<6'), ('PIL', 'Pillow>=11,<12')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import pandas as pd
bruto = pd.read_csv("dados/brutos/catalogo_messy.csv", sep=";", dtype={"codigo_municipio": "string"})
bruto.info()

## 2. Largo e longo

No formato largo, período e tema aparecem nos nomes das colunas. No longo,
essas dimensões viram valores. A transformação altera a unidade da linha: de
documento para combinação documento–tema–período.

In [ ]:
largo = pd.read_csv("dados/brutos/indicadores_largos.csv")
longo = largo.melt(id_vars="id_documento", var_name="tema_periodo", value_name="ocorrencias")
partes = longo["tema_periodo"].str.extract(r"(?P<tema>.+)_(?P<periodo>\d{4})")
longo = pd.concat([longo[["id_documento", "ocorrencias"]], partes], axis=1)
print("Largo:", largo.shape, "| Longo:", longo.shape)
longo.head(6)

## 3. Preservar original, criar versão normalizada

Não sobrescreveremos títulos, municípios ou gêneros. A coluna original permite
auditar o mapa de equivalências e recuperar distinções apagadas por uma regra.
A remoção de acentos pode ajudar correspondência aproximada, mas não deve
substituir automaticamente a grafia de apresentação.

In [ ]:
import re
import unicodedata

def chave_textual(valor):
    if pd.isna(valor):
        return pd.NA
    texto = " ".join(str(valor).strip().lower().split())
    texto = "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")
    return texto

trabalho = bruto.copy()
trabalho["titulo_chave"] = trabalho["titulo"].map(chave_textual)
trabalho["municipio_chave"] = trabalho["municipio"].map(chave_textual)
mapa_generos = {"editorial": "editorial", "notícia": "notícia", "noticia": "notícia", "carta": "carta", "manifesto": "manifesto"}
trabalho["genero_original"] = trabalho["genero"]
trabalho["genero_padronizado"] = trabalho["genero"].map(chave_textual).map(mapa_generos)
trabalho[["titulo", "titulo_chave", "genero_original", "genero_padronizado"]]

## 4. Datas, códigos e ausências

Código de município é identificador textual, não quantidade. Datas parciais
não devem receber dia e mês inventados. `errors='coerce'` transforma falhas em
ausências; isso exige guardar o original e uma razão, pois “desconhecida” é
informação diferente de erro acidental.

In [ ]:
trabalho["data_original"] = trabalho["data_documento"]
trabalho["data_normalizada"] = pd.to_datetime(trabalho["data_documento"], errors="coerce", dayfirst=True)
trabalho["razao_data_ausente"] = pd.NA
falha_data = trabalho["data_normalizada"].isna()
trabalho.loc[falha_data, "razao_data_ausente"] = "data não informada ou não parseável"
trabalho["palavras"] = pd.to_numeric(trabalho["palavras"], errors="coerce")
trabalho["razao_palavras_ausente"] = trabalho["palavras"].isna().map({True: "não contado", False: pd.NA})
trabalho[["data_original", "data_normalizada", "razao_data_ausente", "palavras", "razao_palavras_ausente"]]

## 5. Duplicatas são uma hipótese

IDs repetidos detectam um tipo de duplicata. Registros de um mesmo documento
com IDs diferentes exigem combinação de campos e revisão. Remover pelo título
isolado poderia apagar edições legítimas.

In [ ]:
trabalho["possivel_duplicata"] = trabalho.duplicated(
    subset=["titulo_chave", "data_normalizada", "municipio_chave", "palavras"],
    keep=False,
)
trabalho.loc[trabalho["possivel_duplicata"], ["id_documento", "titulo", "data_original", "palavras"]]

## 6. Relatório e exportação intermediária

Antes/depois deve quantificar transformações, falhas e casos para revisão. A
saída intermediária não substitui os dados brutos.

In [ ]:
relatorio = {
    "linhas": len(trabalho),
    "datas_nao_parseadas": int(trabalho["data_normalizada"].isna().sum()),
    "palavras_ausentes": int(trabalho["palavras"].isna().sum()),
    "generos_sem_mapeamento": int(trabalho["genero_padronizado"].isna().sum()),
    "registros_em_grupos_de_possiveis_duplicatas": int(trabalho["possivel_duplicata"].sum()),
}
trabalho.to_csv("dados/intermediarios/catalogo_normalizado.csv", index=False)
pd.Series(relatorio, name="quantidade")

## Atividade — log de transformação

Registre campo, problema, regra, justificativa, valores afetados, teste,
reversibilidade e responsável. Explique que distinção cada regra pode apagar.
**Log:** Escreva aqui.

## Síntese

Limpeza responsável acrescenta rastreabilidade. Ela não transforma incerteza
substantiva em certeza técnica nem autoriza exclusão silenciosa.